# LSTM Model Training & Evaluation

This notebook trains and evaluates the LSTM model for fake news detection.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import sys
sys.path.insert(0, '../backend')

from models.lstm_model import LSTMModel
from src.trainer import ModelTrainer
from src.data_loader import FakeNewsDataLoader
from src.preprocessor import TextPreprocessor

print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

## 1. Load and Prepare Data

In [ ]:
# Load data
loader = FakeNewsDataLoader()
data = loader.load_data()

preprocessor = TextPreprocessor()
data['processed_text'] = data['text'].apply(preprocessor.preprocess)

# Split data
from sklearn.model_selection import train_test_split
train_data, test_data = train_test_split(
    data, test_size=0.2, random_state=42, stratify=data['label']
)

print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")

## 2. Initialize LSTM Model

In [ ]:
# Model hyperparameters
vocab_size = 10000
embedding_dim = 300
hidden_dim = 128
output_dim = 2  # FAKE or REAL
n_layers = 2
bidirectional = True
dropout = 0.3

# Initialize model
model = LSTMModel(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    output_dim=output_dim,
    n_layers=n_layers,
    bidirectional=bidirectional,
    dropout=dropout
)

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"Model moved to device: {device}")
print(f"\nModel architecture:")
print(model)

## 3. Training Configuration

In [ ]:
# Training parameters
batch_size = 32
num_epochs = 10
learning_rate = 0.001

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, verbose=True
)

# Initialize trainer
trainer = ModelTrainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    model_type='lstm'
)

print(f"Training configuration:")
print(f"  Epochs: {num_epochs}")
print(f"  Batch size: {batch_size}")
print(f"  Learning rate: {learning_rate}")
print(f"  Loss function: {criterion}")

## 4. Train Model

In [ ]:
# Train the model
train_losses, val_losses = trainer.train(
    train_data,
    test_data,
    num_epochs=num_epochs,
    batch_size=batch_size,
    scheduler=scheduler
)

print("Training completed!")

## 5. Loss Visualization

In [ ]:
# Plot loss curves
plt.figure(figsize=(12, 5))
plt.plot(train_losses, label='Training Loss', marker='o')
plt.plot(val_losses, label='Validation Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('LSTM Model - Training and Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Evaluation on Test Set

In [ ]:
# Get predictions
model.eval()
predictions = []
true_labels = []

with torch.no_grad():
    for text, label in test_data:
        # Tokenize and prepare input
        processed = preprocessor.preprocess(text)
        tokens = processed.split()
        
        # Get prediction
        output = model(tokens)
        pred = torch.argmax(output, dim=1).item()
        
        predictions.append(pred)
        true_labels.append(1 if label == 'FAKE' else 0)

# Calculate metrics
accuracy = accuracy_score(true_labels, predictions)
precision = precision_score(true_labels, predictions)
recall = recall_score(true_labels, predictions)
f1 = f1_score(true_labels, predictions)

print("LSTM Model - Test Set Performance:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")

## 7. Confusion Matrix

In [ ]:
import seaborn as sns

cm = confusion_matrix(true_labels, predictions)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['REAL', 'FAKE'],
            yticklabels=['REAL', 'FAKE'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('LSTM Model - Confusion Matrix')
plt.tight_layout()
plt.show()

print(f"\nTrue Negatives:  {cm[0, 0]}")
print(f"False Positives: {cm[0, 1]}")
print(f"False Negatives: {cm[1, 0]}")
print(f"True Positives:  {cm[1, 1]}")

## 8. Save Model

In [ ]:
# Save model
save_path = '../models/lstm_model.pth'
torch.save(model.state_dict(), save_path)
print(f"Model saved to {save_path}")

# Save model config
import json
config = {
    'vocab_size': vocab_size,
    'embedding_dim': embedding_dim,
    'hidden_dim': hidden_dim,
    'output_dim': output_dim,
    'n_layers': n_layers,
    'bidirectional': bidirectional,
    'dropout': dropout
}

with open('../models/lstm_config.json', 'w') as f:
    json.dump(config, f)
print("Model config saved!")